# Studi Model Generatif — D6421 Sesi 07
**Komang Rangga Hariwijaya · NIM 2802493134 · LA01**

Notebook mandiri dari implementasi proyek ini: semua fungsi/class yang dipakai didefinisikan
langsung dalam cell, tanpa import modul proyek atau membaca kode Python eksternal saat runtime.
Penyusunan notebook dan implementasi proyek dibantu AI secara transparan; tinjauan mahasiswa
masih diperlukan. Tidak ada hasil eksperimen yang dibuat-buat.

Urutan: setup → data/split → enam model → training/checkpoint → lima tahap MLP → metrik → lima figur.
**Run All default tidak melakukan training, benchmark, FID ulang, atau regenerasi figur.**
Checkpoint/dataset/hasil tersimpan bersifat opsional; tanpa checkpoint, definisi tetap berjalan
namun inferensi dilewati dengan pesan, bukan diganti bobot acak atau training otomatis.


## 1. Setup dan konfigurasi
Gunakan kernel Python 3.10 pada environment proyek (VS Code: Select Kernel → `.venv`).
Dari terminal, instal `requirements.txt`, lalu jalankan `python -m jupyter lab main.ipynb`.
Jika hanya memiliki notebook ini, dependensi komputasinya dapat dipasang langsung:
```text
python -m pip install torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu128
python -m pip install numpy==2.2.6 matplotlib==3.10.9 scipy==1.15.3 torchmetrics==1.9.0 torch-fidelity==0.4.0 ipykernel jupyterlab
```
CUDA 12.8 ditujukan untuk environment asli; sesuaikan wheel untuk CPU/platform lain dan jangan
bandingkan timing lintas perangkat. Notebook tidak menjalankan instalasi paket otomatis.


In [ ]:
from __future__ import annotations
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import shutil
import statistics
import tempfile
import time
from pathlib import Path

import numpy as np
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import FashionMNIST, MNIST
from torchvision.utils import make_grid, save_image
from torchmetrics.functional.image import structural_similarity_index_measure
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as DisplayImage, Markdown, display


In [ ]:
DEFAULT_CONFIG = {'dataset': 'FashionMNIST',
 'seed': 42,
 'batch_size': 128,
 'num_workers': 0,
 'device': 'auto',
 'precision': 'float32',
 'data_dir': 'data',
 'output_dir': 'artifacts/fashion',
 'latent_dim': 64,
 'noise_dim': 100,
 'sparse_lambda': 0.0001,
 'sparsity_threshold': 0.001,
 'ddpm_steps': 400,
 'ddpm_channels': 32,
 'beta_start': 0.00025,
 'beta_end': 0.05,
 'lr': 0.001,
 'gan_lr': 0.0002,
 'ddpm_lr': 0.0002,
 'epochs': {'mlp': 25,
            'ae': 20,
            'sparse_ae': 20,
            'vae': 25,
            'gan': 40,
            'dcgan': 40,
            'ddpm': 30,
            'decoder': 20},
 'fid_samples': 7000,
 'fid_batch_size': 64,
 'timing_warmup': 3,
 'timing_repeats': 10,
 'normalization': {'ae': '[0,1]',
                   'sparse_ae': '[0,1]',
                   'vae': '[0,1]',
                   'mlp': '[0,1]',
                   'decoder': '[0,1]',
                   'gan': '[-1,1]',
                   'dcgan': '[-1,1]',
                   'ddpm': '[-1,1]'},
 'checkpoint_selection': {'mlp': 'maximum validation accuracy',
                          'ae': 'minimum validation objective',
                          'sparse_ae': 'minimum validation objective',
                          'vae': 'minimum validation ELBO loss',
                          'decoder': 'minimum validation MSE',
                          'ddpm': 'minimum fixed-noise validation MSE',
                          'gan': 'last epoch of predeclared budget',
                          'dcgan': 'last epoch of predeclared budget'}}

IDENTITY = {'name': 'Komang Rangga Hariwijaya',
 'nim': '2802493134',
 'kelas': 'LA01',
 'github_repository': 'https://github.com/Rangga27-hub/tugas2dl-generative-models',
 'review_status': 'pending_user_review'}


In [ ]:
# Ubah pilihan ini, lalu Restart Kernel + Run All.
MODE = "load"                    # "load", "train", atau "resume"
CONFIRM_TRAINING = False          # wajib True agar train/resume benar-benar berjalan
RUN_PILOT = False                # satu epoch DDPM + sampling64 + smoke extractor FID
RUN_SMOKE = False                # dua batch/model; checkpoint sementara, bukan hasil final
SMOKE_FID_EXTRACTOR = True        # hanya aktif jika RUN_SMOKE=True; mungkin unduh bobot Inception
RECOMPUTE_MLP = False             # jalankan lima tahap MLP dari checkpoint terkunci
RECOMPUTE_METRICS = False         # seluruh 7000 test + latensi; tidak otomatis FID
RECOMPUTE_FID = False             # eksplisit; 7000 vs7000 untuk empat generator
REGENERATE_FIGURES = False        # eksplisit; lima figur dihitung ulang dari checkpoint
DISPLAY_SAVED_RESULTS = True
DOWNLOAD_DATA = True              # dipakai hanya saat data benar-benar diperlukan
CHECKPOINTS_TRUSTED = True        # hanya muat checkpoint milik sendiri/yang dipercaya
RETRAIN_OUTPUT_DIR = "artifacts/notebook_fashion"  # tidak menimpa run asli
NOTEBOOK_OUTPUT_DIR = "artifacts/notebook_outputs"

cfg = copy.deepcopy(DEFAULT_CONFIG)
if MODE not in {"load", "train", "resume"}:
    raise ValueError("MODE harus load/train/resume")
if MODE in {"train", "resume"}:
    if Path(RETRAIN_OUTPUT_DIR).resolve() == Path(DEFAULT_CONFIG["output_dir"]).resolve():
        raise ValueError("Training notebook harus memakai output_dir terpisah dari hasil asli")
    cfg["output_dir"] = RETRAIN_OUTPUT_DIR
OUTPUT = Path(NOTEBOOK_OUTPUT_DIR) / MODE
# Hasil terpublikasi hanya ditampilkan untuk mode load dengan konfigurasi asli.
SAVED_RESULTS = Path("results") if MODE == "load" else Path(cfg["output_dir"])
MODEL_NAMES = ("ae", "sparse_ae", "vae", "gan", "dcgan", "ddpm")
ALL_NAMES = ("mlp", "ae", "sparse_ae", "vae", "gan", "dcgan", "ddpm", "decoder")
print("Mode:", MODE, "| Training diizinkan:", CONFIRM_TRAINING)
print("Checkpoint:", cfg["output_dir"], "| Output notebook:", OUTPUT)


In [ ]:
def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, ensure_ascii=False, allow_nan=False)
    os.replace(temp, path)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.set_num_threads(min(8, os.cpu_count() or 1))


def get_device(config):
    name = config.get("device", "auto")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu") if name == "auto" else torch.device(name)


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def environment(device):
    return {"python": platform.python_version(), "platform": platform.platform(),
            "torch": torch.__version__, "torchvision": torchvision.__version__,
            "cuda_runtime": torch.version.cuda, "device": str(device),
            "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
            "precision": "float32", "cudnn_deterministic": torch.backends.cudnn.deterministic}


def rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and state["cuda"] is not None:
        torch.cuda.set_rng_state_all([x.cpu() for x in state["cuda"]])


def atomic_torch_save(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(".tmp")
    torch.save(value, temp)
    os.replace(temp, path)


def load_checkpoint(path, device="cpu"):
    # Only load trusted, locally generated checkpoints. RNG state requires pickle.
    return torch.load(path, map_location=device, weights_only=False)


In [ ]:
seed_all(cfg["seed"])
device = get_device(cfg)
print(json.dumps(environment(device), indent=2))


## 2. Dataset dan split
Fashion-MNIST: gabungan train resmi 60.000 lalu test resmi 10.000. Permutasi NumPy seed42
membentuk 56.000/7.000/7.000; train saja di-shuffle, batch128. Penyimpanan uint8 dikonversi ke
float32 [0,1] per batch; normalisasi GAN/DDPM dilakukan pada jalur model.
Indeks `.npz` lama dipertahankan untuk kecocokan SHA checkpoint. Jika tersedia, indeks publik
`results/protocol/` disalin byte-identik; bila notebook berdiri sendiri, indeks dibuat dari seed yang sama.
Saat memindahkan checkpoint, sertakan **split .npz asli** agar hash juga cocok.


In [ ]:
class DataModule:
    """Pool official 60k + 10k, then persist one random 80/10/10 split."""

    def __init__(self, config, download=True):
        self.config = config
        dataset_cls = {"FashionMNIST": FashionMNIST, "MNIST": MNIST}[config["dataset"]]
        root = Path(config["data_dir"])
        official_train = dataset_cls(root, train=True, download=download)
        official_test = dataset_cls(root, train=False, download=download)
        self.images = torch.cat([official_train.data, official_test.data]).unsqueeze(1)
        self.labels = torch.cat([official_train.targets, official_test.targets])
        assert self.images.shape == (70000, 1, 28, 28)
        self.split_path = root / f'{config["dataset"]}_split_seed{config["seed"]}.npz'
        if not self.split_path.exists():
            order = np.random.default_rng(config["seed"]).permutation(70000)
            np.savez(self.split_path, train=order[:56000], validation=order[56000:63000], test=order[63000:])
        with np.load(self.split_path) as stored:
            self.indices = {k: stored[k].copy() for k in ("train", "validation", "test")}
        expected = (56000, 7000, 7000)
        assert tuple(len(self.indices[k]) for k in self.indices) == expected
        all_indices = np.concatenate(list(self.indices.values()))
        assert np.array_equal(np.sort(all_indices), np.arange(70000)), "Split overlaps or is incomplete"
        self.split_sha256 = hashlib.sha256(self.split_path.read_bytes()).hexdigest()

    def loader(self, split, epoch=0, batch_size=None):
        idx = torch.from_numpy(self.indices[split])
        dataset = TensorDataset(self.images[idx], self.labels[idx], idx)
        generator = torch.Generator().manual_seed(self.config["seed"] + epoch)
        return DataLoader(dataset, batch_size=batch_size or self.config["batch_size"],
                          shuffle=split == "train", num_workers=self.config["num_workers"],
                          pin_memory=torch.cuda.is_available(), generator=generator, drop_last=False)

    def summary(self):
        return {"dataset": self.config["dataset"], "pool_order": "torchvision train then test",
                "split_algorithm": "numpy default_rng(seed).permutation(70000)",
                "seed": self.config["seed"], "sizes": {k: len(v) for k, v in self.indices.items()},
                "split_file": str(self.split_path), "split_sha256": self.split_sha256,
                "pixel_storage": "uint8 [0,255]; converted per batch to float32 [0,1]",
                "disjoint_and_complete": True}


def to_device(batch, device):
    images, labels, indices = batch
    return images.to(device, non_blocking=True).float().div_(255), labels.to(device), indices


In [ ]:
checkpoint_root = Path(cfg["output_dir"])
checkpoint_files = [checkpoint_root / name / filename for name in ALL_NAMES for filename in ("best.pt", "last.pt")]
missing_checkpoints = [str(p) for p in checkpoint_files if not p.exists()]
training_requested = MODE in {"train", "resume"} and CONFIRM_TRAINING
load_available = MODE == "load" and CHECKPOINTS_TRUSTED and not missing_checkpoints
needs_data = load_available or training_requested or RUN_SMOKE or RUN_PILOT

data = None
if needs_data:
    split_name = f'{cfg["dataset"]}_split_seed{cfg["seed"]}.npz'
    destination = Path(cfg["data_dir"]) / split_name
    published_split = Path("results/protocol") / split_name
    if not destination.exists() and published_split.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(published_split, destination)
    data = DataModule(cfg, download=DOWNLOAD_DATA)
    print(data.summary())
    batch_x, batch_y, _ = to_device(next(iter(data.loader("validation"))), device)
    assert tuple(batch_x.shape[1:]) == (1, 28, 28)
    assert 0 <= batch_x.min() <= batch_x.max() <= 1
else:
    print("Dataset belum dimuat: tidak ada pekerjaan komputasi yang diaktifkan.")
if MODE == "load" and missing_checkpoints:
    print("Checkpoint tidak lengkap; tidak training otomatis. Salin best.pt + last.pt dan split asli, atau pilih train secara eksplisit.")
    print("Contoh berkas yang belum ada:", missing_checkpoints[:3])


## 3. Enam model
Arsitektur berikut disalin dari proyek, bukan model pengganti. Vanilla/Sparse AE memiliki kapasitas
identik; perbedaan hanya pada objective training. Definisi MLP dan decoder disertakan untuk tahap berikutnya.


### 3.1 Vanilla AE dan Sparse AE
784 → 256 ReLU → 64 ReLU → 256 ReLU → 784 sigmoid.


In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())

    def forward(self, z):
        return self.net(z).reshape(-1, 1, 28, 28)


class Autoencoder(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                                     nn.Linear(256, latent), nn.ReLU())
        self.decoder = Decoder(latent)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


### 3.2 VAE
Encoder256 dengan kepala mu/logvar64, reparameterisasi Gaussian, decoder sigmoid.


In [ ]:
class VAE(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU())
        self.mu = nn.Linear(256, latent)
        self.logvar = nn.Linear(256, latent)
        self.decoder = Decoder(latent)

    def encode(self, x):
        h = self.encoder(x)
        return self.mu(h), self.logvar(h)

    def forward(self, x, deterministic=False):
        mu, logvar = self.encode(x)
        z = mu if deterministic else mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decoder(z), mu, logvar


### 3.3 Vanilla GAN
Prior Gaussian100; generator dan discriminator MLP; keluaran tanh dan satu logit D.


In [ ]:
class VanillaGAN(nn.Module):
    def __init__(self, noise=100):
        super().__init__()
        self.generator = nn.Sequential(nn.Linear(noise, 256), nn.LeakyReLU(0.2),
                                       nn.Linear(256, 512), nn.LeakyReLU(0.2),
                                       nn.Linear(512, 784), nn.Tanh(), nn.Unflatten(1, (1, 28, 28)))
        self.discriminator = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.LeakyReLU(0.2),
                                           nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Linear(256, 1))

    def forward(self, z):
        return self.generator(z)


### 3.4 DCGAN
Proyeksi128×7×7 → transposed-conv64×14×14 → 1×28×28; D stride2.


In [ ]:
class DCGAN(nn.Module):
    def __init__(self, noise=100):
        super().__init__()
        self.generator = nn.Sequential(
            nn.Linear(noise, 128 * 7 * 7), nn.Unflatten(1, (128, 7, 7)),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 1, 4, 2, 1), nn.Tanh())
        self.discriminator = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Flatten(), nn.Linear(128 * 7 * 7, 1))
        self.apply(self._init)

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
            nn.init.normal_(module.weight, 0, 0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.normal_(module.weight, 1, 0.02)
            nn.init.zeros_(module.bias)

    def forward(self, z):
        return self.generator(z)


### 3.5 DDPM
U-Net ringan 28→14→7→14→28, timestep sinusoidal; konfigurasi aktual T400, beta0,00025..0,05.


In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.net = nn.Sequential(nn.Linear(dim, dim * 2), nn.SiLU(), nn.Linear(dim * 2, dim))

    def forward(self, t):
        freq = torch.exp(torch.arange(self.dim // 2, device=t.device) * (-math.log(10000) / (self.dim // 2 - 1)))
        angles = t.float()[:, None] * freq[None]
        return self.net(torch.cat([angles.sin(), angles.cos()], dim=1))


class TimeBlock(nn.Module):
    def __init__(self, inp, out, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(inp, out, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out)
        self.time = nn.Linear(time_dim, out)
        self.conv2 = nn.Conv2d(out, out, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out)
        self.skip = nn.Conv2d(inp, out, 1) if inp != out else nn.Identity()

    def forward(self, x, emb):
        h = F.silu(self.norm1(self.conv1(x))) + self.time(emb)[:, :, None, None]
        h = F.silu(self.norm2(self.conv2(h)))
        return h + self.skip(x)


class Denoiser(nn.Module):
    """Small U-Net: 28 -> 14 -> 7 -> 14 -> 28; timestep injection in every block."""
    def __init__(self, channels=32):
        super().__init__()
        c = channels
        self.time = TimeEmbedding(c * 4)
        self.down1 = TimeBlock(1, c, c * 4)
        self.down2 = TimeBlock(c, c * 2, c * 4)
        self.mid = TimeBlock(c * 2, c * 4, c * 4)
        self.up2 = TimeBlock(c * 6, c * 2, c * 4)
        self.up1 = TimeBlock(c * 3, c, c * 4)
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x, t):
        e = self.time(t)
        a = self.down1(x, e)
        b = self.down2(F.avg_pool2d(a, 2), e)
        h = self.mid(F.avg_pool2d(b, 2), e)
        h = self.up2(torch.cat([F.interpolate(h, size=b.shape[-2:], mode="nearest"), b], 1), e)
        h = self.up1(torch.cat([F.interpolate(h, size=a.shape[-2:], mode="nearest"), a], 1), e)
        return self.out(h)


class DDPM(nn.Module):
    def __init__(self, steps=200, channels=32, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.denoiser = Denoiser(channels)
        self.steps = steps
        beta = torch.linspace(beta_start, beta_end, steps)
        alpha = 1 - beta
        abar = torch.cumprod(alpha, 0)
        prev = F.pad(abar[:-1], (1, 0), value=1)
        for name, value in {"beta": beta, "alpha": alpha, "abar": abar,
                            "posterior_variance": beta * (1 - prev) / (1 - abar)}.items():
            self.register_buffer(name, value)

    def q_sample(self, x, t, noise):
        a = self.abar[t, None, None, None]
        return a.sqrt() * x + (1 - a).sqrt() * noise

    def forward(self, x, t):
        return self.denoiser(x, t)

    @torch.no_grad()
    def sample(self, x, generator=None, snapshots=False):
        chain = {self.steps: x[:1].detach().cpu()} if snapshots else {}
        keep = {0, self.steps // 16, self.steps // 4, self.steps // 2, 3 * self.steps // 4}
        for i in reversed(range(self.steps)):
            t = torch.full((len(x),), i, device=x.device, dtype=torch.long)
            eps = self.denoiser(x, t)
            # Predict/clamp x0 and use the exact DDPM posterior mean.
            x0 = ((x - (1 - self.abar[i]).sqrt() * eps) / self.abar[i].sqrt()).clamp(-1, 1)
            prev = self.abar[i - 1] if i else torch.ones_like(self.abar[i])
            mean = (prev.sqrt() * self.beta[i] / (1 - self.abar[i])) * x0
            mean += ((1 - prev) * self.alpha[i].sqrt() / (1 - self.abar[i])) * x
            noise = torch.randn(x.shape, device=x.device, generator=generator) if i else torch.zeros_like(x)
            x = mean + self.posterior_variance[i].sqrt() * noise
            if snapshots and i in keep:
                chain[i] = x[:1].detach().cpu()
        return (x, chain) if snapshots else x


### 3.6 MLP dan pemilih model
Classifier784→256→64→10. Proyeksi transpose sengaja tanpa bias sesuai eksperimen.


In [ ]:
class Classifier(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, latent)
        self.fc3 = nn.Linear(latent, 10)

    def encode(self, x):
        return F.relu(self.fc2(F.relu(self.fc1(x.flatten(1)))))

    def forward(self, x):
        return self.fc3(self.encode(x))

    def transpose_latent(self, z):
        # For row vectors, z @ W2 then @ W1 is the column-vector W1^T ReLU(W2^T z).
        return torch.sigmoid(F.relu(z @ self.fc2.weight) @ self.fc1.weight).reshape(-1, 1, 28, 28)

    def transpose_class(self, one_hot):
        return self.transpose_latent(F.relu(one_hot @ self.fc3.weight))


def build_model(name, config):
    latent, noise = config["latent_dim"], config["noise_dim"]
    if name in ("ae", "sparse_ae"):
        return Autoencoder(latent)
    if name == "vae":
        return VAE(latent)
    if name == "mlp":
        return Classifier(latent)
    if name == "decoder":
        return Decoder(latent)
    if name == "gan":
        return VanillaGAN(noise)
    if name == "dcgan":
        return DCGAN(noise)
    if name == "ddpm":
        return DDPM(config["ddpm_steps"], config["ddpm_channels"], config["beta_start"], config["beta_end"])
    raise ValueError(name)


## 4. Training dan checkpoint
AE: MSE mean piksel; Sparse: MSE + lambda mean-batch sum-laten |z|.
VAE: BCE sum784 + KL sum64, mean batch. GAN: BCEWithLogits, G non-saturating.
DDPM: noise MSE; classifier: cross-entropy logits; frozen decoder: MSE.
Adam/LR/betas, pemilihan validation, seed, timing train+validation dan resume RNG sama dengan proyek.
Default hanya memuat checkpoint lokal tepercaya; pickle checkpoint jangan dimuat dari sumber sembarang.


In [ ]:
NAMES = ALL_NAMES

def objective(name, model, x, y, cfg, classifier=None):
    if name in ("ae", "sparse_ae"):
        pred, z = model(x)
        mse = F.mse_loss(pred, x)
        penalty = z.abs().sum(1).mean()
        loss = mse + (cfg["sparse_lambda"] * penalty if name == "sparse_ae" else 0)
        return loss, {"mse": mse, "latent_l1_sum": penalty}
    if name == "vae":
        pred, mu, logvar = model(x)
        recon = F.binary_cross_entropy(pred, x, reduction="none").flatten(1).sum(1).mean()
        kl = (-0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(1)).mean()
        return recon + kl, {"recon_bce_sum": recon, "kl_sum": kl}
    if name == "mlp":
        logits = model(x)
        return F.cross_entropy(logits, y), {"accuracy": (logits.argmax(1) == y).float().mean()}
    if name == "decoder":
        with torch.no_grad():
            z = classifier.encode(x)
        return F.mse_loss(model(z), x), {}
    if name == "ddpm":
        t = torch.randint(model.steps, (len(x),), device=x.device)
        noise = torch.randn_like(x)
        xt = model.q_sample(x * 2 - 1, t, noise)
        return F.mse_loss(model(xt, t), noise), {}
    raise ValueError(name)


def gan_step(model, x, optimizers, cfg, train):
    n = len(x)
    opt_g, opt_d = optimizers
    real = x * 2 - 1
    z = torch.randn(n, cfg["noise_dim"], device=x.device)
    if train:
        opt_d.zero_grad(set_to_none=True)
    fake = model.generator(z)
    real_logits, fake_logits = model.discriminator(real), model.discriminator(fake.detach())
    d_loss = F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
    d_loss += F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits))
    if train:
        d_loss.backward()
        opt_d.step()
        opt_g.zero_grad(set_to_none=True)
        model.discriminator.requires_grad_(False)
    logits = model.discriminator(fake)
    g_loss = F.binary_cross_entropy_with_logits(logits, torch.ones_like(logits))
    if train:
        g_loss.backward()
        opt_g.step()
        model.discriminator.requires_grad_(True)
    return g_loss, {"g_loss": g_loss, "d_loss": d_loss}


def run_epoch(name, model, loader, device, cfg, optimizers, classifier=None, train=True, max_batches=None):
    model.train(train)
    totals, count = {}, 0
    with torch.set_grad_enabled(train):
        for batch_index, batch in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            x, y, _ = to_device(batch, device)
            if name in ("gan", "dcgan"):
                loss, parts = gan_step(model, x, optimizers, cfg, train)
            else:
                if train:
                    optimizers[0].zero_grad(set_to_none=True)
                loss, parts = objective(name, model, x, y, cfg, classifier)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite {name} loss")
                if train:
                    loss.backward()
                    optimizers[0].step()
            values = {"loss": loss, **parts}
            for key, value in values.items():
                number = value.detach().item()
                if not torch.isfinite(torch.tensor(number)):
                    raise FloatingPointError(f"Non-finite {name}/{key}")
                totals[key] = totals.get(key, 0.0) + number * len(x)
            count += len(x)
    if count == 0:
        raise ValueError("Empty epoch")
    return {k: v / count for k, v in totals.items()}


def train_model(name, cfg, data=None, resume=False, epochs=None, max_batches=None):
    seed_all(cfg["seed"])
    device = get_device(cfg)
    root = Path(cfg["output_dir"])
    run = root / name
    run.mkdir(parents=True, exist_ok=True)
    data = data or DataModule(cfg)
    model = build_model(name, cfg).to(device)
    classifier, classifier_hash = None, None
    if name == "decoder":
        classifier_path = root / "mlp" / "best.pt"
        saved = load_checkpoint(classifier_path)
        if saved["split_sha256"] != data.split_sha256:
            raise ValueError("Classifier split mismatch")
        classifier_hash = hashlib.sha256(classifier_path.read_bytes()).hexdigest()
        classifier = build_model("mlp", cfg).to(device)
        classifier.load_state_dict(saved["model"])
        classifier.eval().requires_grad_(False)
    if name in ("gan", "dcgan"):
        optimizers = [torch.optim.Adam(model.generator.parameters(), lr=cfg["gan_lr"], betas=(0.5, 0.999)),
                      torch.optim.Adam(model.discriminator.parameters(), lr=cfg["gan_lr"], betas=(0.5, 0.999))]
    else:
        lr = cfg["ddpm_lr"] if name == "ddpm" else cfg["lr"]
        optimizers = [torch.optim.Adam(model.parameters(), lr=lr)]
    target = epochs if epochs is not None else cfg["epochs"][name]
    if target < 1:
        raise ValueError("Epochs must be positive")
    start, best, history = 0, float("inf"), []
    last = run / "last.pt"
    if last.exists() and not resume:
        raise FileExistsError(f"{last} exists; use --resume or another output_dir")
    if resume:
        saved = load_checkpoint(last)
        if saved["name"] != name or saved["split_sha256"] != data.split_sha256 or saved["config"] != cfg:
            raise ValueError("Resume requires identical model, config and split; use --epochs to extend budget")
        if saved.get("classifier_sha256") != classifier_hash:
            raise ValueError("Frozen classifier changed; cannot resume decoder")
        if saved["max_batches"] != max_batches:
            raise ValueError("Cannot mix smoke and full epochs")
        model.load_state_dict(saved["model"])
        for opt, state in zip(optimizers, saved["optimizers"]):
            opt.load_state_dict(state)
        start, best, history = saved["epoch"], saved["best_score"], saved["history"]
        restore_rng(saved["rng"])
    env = environment(device)
    save_json(run / "environment.json", env)
    save_json(run / "config.json", cfg)
    save_json(root / "data.json", data.summary())
    fixed_gen = torch.Generator(device=device).manual_seed(cfg["seed"] + 9000)
    fixed_z = torch.randn(16, cfg["noise_dim"], device=device, generator=fixed_gen)
    for epoch in range(start, target):
        sync(device)
        begin = time.perf_counter()
        train_metrics = run_epoch(name, model, data.loader("train", epoch), device, cfg, optimizers,
                                  classifier, True, max_batches)
        # Fixed validation random draws across epochs, without changing training RNG.
        devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == "cuda" else []
        with torch.random.fork_rng(devices=devices):
            torch.manual_seed(cfg["seed"] + 555)
            if device.type == "cuda":
                torch.cuda.manual_seed_all(cfg["seed"] + 555)
            val_metrics = run_epoch(name, model, data.loader("validation"), device, cfg, optimizers,
                                    classifier, False, max_batches)
        sync(device)
        seconds = time.perf_counter() - begin
        score = -val_metrics["accuracy"] if name == "mlp" else val_metrics["loss"]
        improved = score < best or name in ("gan", "dcgan")
        if improved:
            best = score
        record = {"epoch": epoch + 1, "seconds_train_validation": seconds,
                  **{"train_" + k: v for k, v in train_metrics.items()},
                  **{"val_" + k: v for k, v in val_metrics.items()}}
        history.append(record)
        state = {"name": name, "model": model.state_dict(), "optimizers": [o.state_dict() for o in optimizers],
                 "epoch": epoch + 1, "best_score": best, "config": cfg, "history": history,
                 "rng": rng_state(), "environment": env, "split_sha256": data.split_sha256,
                 "max_batches": max_batches, "classifier_sha256": classifier_hash,
                 "timing_scope": "train+validation; excludes checkpoint/log/sample I/O"}
        atomic_torch_save(state, last)
        if improved:
            atomic_torch_save(state, run / "best.pt")
        save_json(run / "history.json", history)
        with (run / "history.csv").open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(history[0]))
            writer.writeheader()
            writer.writerows(history)
        if name in ("gan", "dcgan"):
            model.eval()
            with torch.no_grad():
                samples = model(fixed_z).add(1).div(2).clamp(0, 1)
            save_image(samples, run / f"samples_epoch_{epoch+1:03d}.png", nrow=4)
        print(f"{name} epoch {epoch+1}/{target} {seconds:.2f}s train={train_metrics} val={val_metrics}", flush=True)
    return model


In [ ]:
def load_model(name, cfg, device, data=None):
    path = Path(cfg["output_dir"]) / name / "best.pt"
    saved = load_checkpoint(path)
    if saved["name"] != name or saved["config"] != cfg:
        raise ValueError(f"Configuration mismatch in {path}")
    if saved["max_batches"] is not None:
        raise ValueError("Smoke checkpoints cannot be used for final evaluation")
    if data is not None and saved["split_sha256"] != data.split_sha256:
        raise ValueError("Split mismatch")
    model = build_model(name, cfg).to(device)
    model.load_state_dict(saved["model"])
    model.eval().requires_grad_(False)
    last = load_checkpoint(path.parent / "last.pt")
    if last["config"] != cfg or last["split_sha256"] != saved["split_sha256"]:
        raise ValueError("Final/selected checkpoint provenance mismatch")
    metadata = {"checkpoint": str(path), "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                "selected_epoch": saved["epoch"], "completed_epochs": last["epoch"],
                "mean_seconds_epoch": statistics.mean(x["seconds_train_validation"] for x in last["history"]),
                "timing_scope": saved["timing_scope"], "selection": cfg["checkpoint_selection"][name]}
    if name == "decoder":
        mlp_path = Path(cfg["output_dir"]) / "mlp" / "best.pt"
        if saved["classifier_sha256"] != hashlib.sha256(mlp_path.read_bytes()).hexdigest():
            raise ValueError("Decoder belongs to another classifier checkpoint")
    return model, metadata


In [ ]:
def benchmark_ddpm(cfg, data, device, max_batches=None, skip_fid=False):
    model = build_model("ddpm", cfg).to(device)
    opt = [torch.optim.Adam(model.parameters(), lr=cfg["ddpm_lr"])]
    # Warm-up is separate from the timed epoch and does not produce a final checkpoint.
    run_epoch("ddpm", model, data.loader("train"), device, cfg, opt, train=True, max_batches=2)
    sync(device)
    start = time.perf_counter()
    metrics = run_epoch("ddpm", model, data.loader("train"), device, cfg, opt, train=True, max_batches=max_batches)
    sync(device)
    epoch_seconds = time.perf_counter() - start
    model.eval()
    gen = torch.Generator(device=device).manual_seed(cfg["seed"] + 15000)
    noise = make_noise("ddpm", 64, cfg, device, gen)
    sync(device)
    start = time.perf_counter()
    images = sample("ddpm", model, noise, gen)
    sync(device)
    sampling_seconds = time.perf_counter() - start
    batches = len(data.loader("train"))
    measured_batches = min(max_batches, batches) if max_batches else batches
    estimated_epoch_seconds = epoch_seconds * batches / measured_batches
    result = {"environment": environment(device), "config": cfg, "split_sha256": data.split_sha256,
              "measured_train_batches": measured_batches, "full_train_batches": batches,
              "full_epoch_measured": measured_batches == batches, "training_loss": metrics,
              "measured_training_seconds": epoch_seconds, "estimated_full_train_epoch_seconds": estimated_epoch_seconds,
              "train_timing_scope": "train only; excludes validation/I/O; two warm-up batches already executed",
              "reverse_sampling_seconds_batch64": sampling_seconds,
              "estimated_ddpm_training_seconds": estimated_epoch_seconds * cfg["epochs"]["ddpm"],
              "estimated_ddpm_fid_sampling_seconds": sampling_seconds * math.ceil(cfg["fid_samples"] / 64),
              "ddpm_steps": model.steps, "terminal_alpha_bar": model.abar[-1].item(),
              "estimation_limit": "single timing; validation, extractor time, and thermal variation add overhead"}
    if not skip_fid:
        metric = build_fid(device)
        x, _, _ = to_device(next(iter(data.loader("validation", batch_size=64))), device)
        sync(device)
        start = time.perf_counter()
        with torch.inference_mode():
            metric.update(x.repeat(1, 3, 1, 1), real=True)
            metric.update(images.repeat(1, 3, 1, 1), real=False)
        sync(device)
        result["fid_extractor_smoke"] = {"passed": True, "n_each": 64,
                                         "update_seconds_real_and_fake": time.perf_counter() - start,
                                         "note": "feature path test, not a final FID estimate"}
    save_json(Path(cfg["output_dir"]) / "benchmark_ddpm.json", result)
    return result


### 4.1 Pilot dan uji mini
Helper sampling/FID minimum diletakkan di sini karena diperlukan pilot dan smoke; penjelasan serta
penghitungan metrik lengkap tetap pada Bagian6. Pilot/tes tidak berjalan tanpa flag eksplisit.
Smoke memakai arsitektur, batch dan T asli, tetapi hanya dua batch train/validation per model.
Hasilnya tidak memenuhi target akurasi dan **bukan** hasil final.


In [ ]:
@torch.inference_mode()
def sample(name, model, noise, generator=None):
    if name == "vae":
        return model.decoder(noise).clamp(0, 1)
    if name in ("gan", "dcgan"):
        return model(noise).add(1).div(2).clamp(0, 1)
    if name == "ddpm":
        return model.sample(noise, generator=generator).add(1).div(2).clamp(0, 1)
    raise ValueError(name)


def make_noise(name, n, cfg, device, generator):
    shape = (n, 1, 28, 28) if name == "ddpm" else (n, cfg["latent_dim"] if name == "vae" else cfg["noise_dim"])
    return torch.randn(shape, device=device, generator=generator)


def build_fid(device):
    from torchmetrics.image.fid import FrechetInceptionDistance
    # float32 RGB in [0,1], internally converted to uint8 by normalize=True.
    # torch-fidelity compatible Inception-v3 features, no manual resize.
    return FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).to(device)


In [ ]:
def notebook_smoke_test(cfg, data, device, fid_extractor=True):
    parent = Path("artifacts/notebook_smoke")
    parent.mkdir(parents=True, exist_ok=True)
    report = {"purpose": "bounded smoke only, not final experimental metrics", "models": {}}
    with tempfile.TemporaryDirectory(prefix="run_", dir=parent) as temporary:
        smoke_cfg = copy.deepcopy(cfg)
        smoke_cfg["output_dir"] = temporary
        for name in ALL_NAMES:
            seed_all(smoke_cfg["seed"])
            initial = build_model(name, smoke_cfg)
            before = {k: v.clone() for k, v in initial.named_parameters()}
            del initial
            model = train_model(name, smoke_cfg, data=data, epochs=1, max_batches=2)
            changed = any(not torch.equal(before[k], v.detach().cpu()) for k, v in model.named_parameters())
            assert changed, f"{name}: bobot tidak berubah"
            checkpoint = load_checkpoint(Path(temporary) / name / "last.pt")
            assert checkpoint["epoch"] == 1 and checkpoint["max_batches"] == 2
            assert all(math.isfinite(float(v)) for k, v in checkpoint["history"][-1].items())
            model.eval()
            x, _, _ = to_device(next(iter(data.loader("validation", batch_size=4))), device)
            with torch.inference_mode():
                if name in ("ae", "sparse_ae"):
                    output = model(x)[0]
                elif name == "vae":
                    output = model(x, deterministic=True)[0]
                elif name == "mlp":
                    output = model(x)
                    assert model.transpose_class(torch.eye(10, device=device)).shape == (10, 1, 28, 28)
                elif name == "decoder":
                    encoder = build_model("mlp", cfg).to(device).eval().requires_grad_(False)
                    encoder.load_state_dict(load_checkpoint(Path(temporary)/"mlp/best.pt")["model"])
                    output = model(encoder.encode(x))
                else:
                    gen = torch.Generator(device=device).manual_seed(cfg["seed"])
                    output = sample(name, model, make_noise(name, 4, cfg, device, gen), gen)
                assert torch.isfinite(output).all()
                expected = (4, 10) if name == "mlp" else (4, 1, 28, 28)
                assert tuple(output.shape) == expected
            report["models"][name] = {"parameter_update": changed, "finite_loss": True,
                                       "output_shape": list(output.shape), "checkpoint_reload": True}
            del model
        train_model("ae", smoke_cfg, data=data, resume=True, epochs=2, max_batches=2)
        assert load_checkpoint(Path(temporary)/"ae/last.pt")["epoch"] == 2
        report["resume_ae_epoch2"] = True
        if fid_extractor:
            metric = build_fid(device)
            with torch.inference_mode():
                metric.update(x.repeat(1, 3, 1, 1), real=True)
                metric.update(output.repeat(1, 3, 1, 1).clamp(0, 1), real=False)
            report["fid_extractor"] = "real/fake update passed; no low-N FID reported"
    return report

smoke_report = None
if RUN_SMOKE:
    smoke_report = notebook_smoke_test(cfg, data, device, SMOKE_FID_EXTRACTOR)
    display(smoke_report)
if RUN_PILOT:
    pilot_report = benchmark_ddpm(cfg, data, device)
    display(pilot_report)


In [ ]:
models, metadata = {}, {}
if training_requested:
    if not CHECKPOINTS_TRUSTED:
        raise ValueError("Training decoder/resume juga memuat checkpoint lokal; konfirmasi kepercayaan dahulu")
    pilot_path = Path(cfg["output_dir"]) / "benchmark_ddpm.json"
    if not pilot_path.exists():
        raise RuntimeError("Aktifkan RUN_PILOT dan periksa hasil pilot sebelum full training")
    pilot = json.loads(pilot_path.read_text(encoding="utf-8"))
    if (pilot["config"] != cfg or pilot["split_sha256"] != data.split_sha256
        or not pilot["full_epoch_measured"] or not pilot.get("fid_extractor_smoke", {}).get("passed")):
        raise ValueError("Pilot harus full epoch, config/split identik, dan extractor FID lolos")
    if MODE == "train" and any((Path(cfg["output_dir"])/n/"last.pt").exists() for n in ALL_NAMES):
        raise FileExistsError("Output training sudah berisi checkpoint; pilih resume atau folder run baru")
    for name in ("mlp", "ddpm", "ae", "sparse_ae", "vae", "gan", "dcgan", "decoder"):
        last = Path(cfg["output_dir"]) / name / "last.pt"
        train_model(name, cfg, data=data, resume=MODE == "resume" and last.exists())
    load_available = True
elif MODE in {"train", "resume"}:
    print("Training/resume dilewati: CONFIRM_TRAINING masih False.")

if load_available:
    for name in ALL_NAMES:
        models[name], metadata[name] = load_model(name, cfg, device, data)
    print("Checkpoint tervalidasi:", {n: m["selected_epoch"] for n, m in metadata.items()})
else:
    print("Belum ada model terlatih di memori; tidak ada fallback ke bobot acak.")


## 5. Lima eksperimen MLP
1. Classifier dan akurasi test, W1/W2/W3 serta aktivasi ReLU64.
2. One-hot kelas → W3ᵀ → ReLU → W2ᵀ → ReLU → W1ᵀ → sigmoid.
3. Fitur test → W2ᵀ → ReLU → W1ᵀ → sigmoid.
4. Decoder64→256→784 memakai classifier beku, pembanding Vanilla AE laten64.
5. Gaussian diagonal dengan mean/std **populasi** fitur test; hanya analisis akhir setelah decoder terkunci.

Pengumpulan fitur dan sampling di bawah mengikuti kode proyek. MSE/PSNR/SSIM pembanding dihitung
pada Bagian6 dengan rumus yang sama; angka tersimpan lama tidak diklaim sebagai hitung ulang.


In [ ]:
@torch.inference_mode()
def collect_mlp_artifacts(cfg, data, device, models, metadata, output_root):
    seed_all(cfg["seed"])
    mlp, decoder, ae = (models[n] for n in ("mlp", "decoder", "ae"))
    output = Path(output_root) / "mlp_experiments"
    output.mkdir(parents=True, exist_ok=True)
    codes, labels, ids = [], [], []
    correct, total = 0, 0
    for batch in data.loader("test"):
        x, y, idx = to_device(batch, device)
        z = mlp.encode(x)
        correct += (mlp.fc3(z).argmax(1) == y).sum().item()
        total += len(x)
        codes.append(z.cpu())
        labels.append(y.cpu())
        ids.append(idx)
    codes, labels, ids = torch.cat(codes), torch.cat(labels), torch.cat(ids)
    # Population std, not the unbiased sample estimator; diagonal approximation.
    mu, sigma = codes.mean(0), codes.std(0, correction=0)
    generator = torch.Generator(device=device).manual_seed(cfg["seed"] + 14000)
    random_z = mu.to(device) + sigma.to(device) * torch.randn(16, cfg["latent_dim"], device=device, generator=generator)
    first = data.images[ids[:10]].to(device).float().div(255)
    one_hot = torch.eye(10, device=device)
    artifacts = {
        "indices": ids, "test_labels": labels, "test_latent": codes,
        "latent_mu": mu, "latent_sigma_population": sigma,
        "W1": mlp.fc1.weight.cpu(), "W2": mlp.fc2.weight.cpu(), "W3": mlp.fc3.weight.cpu(),
        "class_transpose": mlp.transpose_class(one_hot).cpu(),
        "originals": first.cpu(), "latent_transpose": mlp.transpose_latent(codes[:10].to(device)).cpu(),
        "frozen_reconstruction": decoder(codes[:10].to(device)).cpu(),
        "ae_reconstruction": ae(first)[0].cpu(), "diagonal_gaussian_samples": decoder(random_z).cpu(),
        "sampling_seed": cfg["seed"] + 14000,
        "test_statistics_warning": "Final analysis only; test latent statistics not used for model/checkpoint selection or generator FID"}
    atomic_torch_save(artifacts, output / "artifacts.pt")
    result = {"test_accuracy": correct/total, "target_accuracy": 0.88, "target_met": correct/total >= 0.88,
              "n_test": total, "weight_shapes": {k: list(artifacts[k].shape) for k in ("W1", "W2", "W3")},
              "classifier": metadata["mlp"], "decoder": metadata["decoder"], "vanilla_ae": metadata["ae"],
              "split_sha256": data.split_sha256, "sampling_seed": cfg["seed"]+14000,
              "figure_test_indices": ids[:10].tolist(), "transpose_bias": "not used as specified",
              "sampling_distribution": "diagonal Gaussian of test ReLU features; population std; locked decoder"}
    save_json(output / "results.json", result)
    return artifacts, result


In [ ]:
mlp, mlp_results = None, None
if models and (RECOMPUTE_MLP or REGENERATE_FIGURES or RECOMPUTE_METRICS):
    mlp, mlp_results = collect_mlp_artifacts(cfg, data, device, models, metadata, OUTPUT)
    print("Akurasi test MLP:", mlp_results["test_accuracy"], "| Target tercapai:", mlp_results["target_met"])
elif DISPLAY_SAVED_RESULTS:
    saved_mlp = SAVED_RESULTS / "mlp_experiments/results.json"
    if saved_mlp.exists():
        display(Markdown(f"**Hasil MLP tersimpan, bukan hitung ulang:** `{saved_mlp}`"))
        display(json.loads(saved_mlp.read_text(encoding="utf-8")))
    else:
        print("Eksperimen MLP belum dijalankan; aktifkan RECOMPUTE_MLP setelah checkpoint lengkap.")


## 6. Evaluasi metrik
MSE/PSNR/SSIM: mean per citra seluruh7000 test [0,1], SSIM Gaussian11 sigma1,5,
PSNR MSE-nol dicap float32 tiny. FID: Inception-v3 2048 torch-fidelity, grayscale diulang RGB,
normalize=True; **7000 sintetis vs7000 test** untuk empat generator, subset identik.
Latensi: batch64, eval/no-grad, tiga warm-up, median10 ulangan, sinkronisasi CUDA; DDPM termasuk
reverse chain400 langkah. FID/latensi berat hanya aktif lewat flag eksplisit.
Timing/FID yang dihitung ulang disimpan terpisah, tidak menimpa angka publikasi asli.


In [ ]:
GENERATORS = ("vae", "gan", "dcgan", "ddpm")
RECONSTRUCTORS = ("ae", "sparse_ae", "vae", "decoder")

@torch.inference_mode()
def reconstruct(name, model, x, classifier=None):
    if name in ("ae", "sparse_ae"):
        return model(x)[0]
    if name == "vae":
        return model(x, deterministic=True)[0]
    if name == "decoder":
        return model(classifier.encode(x))
    raise ValueError(name)


def reconstruction_scores(pred, target):
    mse = (pred - target).square().flatten(1).mean(1)
    # Exact zero uses float32 tiny to produce a finite, explicitly capped PSNR.
    psnr = -10 * torch.log10(mse.clamp_min(torch.finfo(torch.float32).tiny))
    ssim = structural_similarity_index_measure(pred, target, data_range=1.0, reduction="none")
    return mse, psnr, ssim


@torch.inference_mode()
def reconstruction_metrics(name, model, data, device, cfg, classifier=None):
    sums = torch.zeros(3, dtype=torch.float64, device=device)
    n, zero_count, sparse_count, sparse_total = 0, 0, 0, 0
    for batch in data.loader("test"):
        x, _, _ = to_device(batch, device)
        pred = reconstruct(name, model, x, classifier).clamp(0, 1)
        values = reconstruction_scores(pred, x)
        sums += torch.stack([v.double().sum() for v in values])
        zero_count += (values[0] == 0).sum().item()
        n += len(x)
        if name in ("ae", "sparse_ae"):
            z = model.encoder(x)
            sparse_count += (z.abs() < cfg["sparsity_threshold"]).sum().item()
            sparse_total += z.numel()
    result = dict(zip(("MSE", "PSNR", "SSIM"), (sums / n).cpu().tolist()))
    result.update({"n_test": n, "zero_mse_images": zero_count, "psnr_zero_policy": "clamp MSE to float32 tiny",
                   "aggregation": "mean of per-image metrics; SSIM Gaussian 11x11 sigma1.5, data_range=1"})
    if sparse_total:
        result.update({"latent_near_zero_fraction": sparse_count / sparse_total,
                       "latent_near_zero_threshold": cfg["sparsity_threshold"]})
    return result


@torch.inference_mode()
def latency(name, model, cfg, device, real_batch=None):
    generator = torch.Generator(device=device).manual_seed(cfg["seed"] + 12000)
    value = real_batch[:64] if name in ("ae", "sparse_ae") else make_noise(name, 64, cfg, device, generator)
    if len(value) != 64:
        raise ValueError("Timing needs exactly 64 images")
    def operation():
        return model(value)[0] if name in ("ae", "sparse_ae") else sample(name, model, value, generator)
    for _ in range(cfg["timing_warmup"]):
        operation()
    times = []
    for _ in range(cfg["timing_repeats"]):
        sync(device)
        start = time.perf_counter()
        operation()
        sync(device)
        times.append((time.perf_counter() - start) * 1000)
    return {"median_ms_batch64": statistics.median(times), "all_ms": times,
            "warmup": cfg["timing_warmup"], "repeats": cfg["timing_repeats"],
            "scope": "AE reconstruction / generators from prepared initial noise; DDPM includes full reverse chain and per-step RNG; no H2D or disk I/O",
            "ddpm_steps": cfg["ddpm_steps"] if name == "ddpm" else None}


def write_table(cfg):
    root = Path(cfg["output_dir"]) / "evaluation"
    notes_path = Path(cfg["output_dir"]) / "visual_notes.json"
    if notes_path.exists():
        import json
        notes = json.loads(notes_path.read_text(encoding="utf-8"))
    else:
        notes = {}
    rows = []
    fid_n = set()
    for name in ("ae", "sparse_ae", "vae", "gan", "dcgan", "ddpm"):
        path = root / f"{name}.json"
        if not path.exists():
            continue
        import json
        result = json.loads(path.read_text(encoding="utf-8"))
        fid = result.get("fid", {})
        if fid:
            fid_n.add((fid["n_real"], fid["subset_sha256"]))
        row = {"model": name, "latent_dim": "1x28x28" if name == "ddpm" else cfg["latent_dim"] if name in RECONSTRUCTORS else cfg["noise_dim"],
               "seconds_epoch": result["checkpoint"]["mean_seconds_epoch"],
               "ms_batch64": result["latency"]["median_ms_batch64"],
               **{key: result.get("reconstruction", {}).get(key, "N/A") for key in ("MSE", "PSNR", "SSIM")},
               "FID": fid.get("FID", "not measured" if name in GENERATORS else "N/A"),
               "FID_N": fid.get("n_real", "N/A"), "stability": notes.get(name) or "requires curve and sample inspection"}
        rows.append(row)
    if len(fid_n) > 1:
        raise ValueError("Do not compare FID with inconsistent N/subsets; reevaluate all generators")
    if rows:
        with (root / "comparison.csv").open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)


In [ ]:
@torch.inference_mode()
def fid_metrics(name, model, data, cfg, device, n, output_root=None):
    if not 2 <= n <= 7000:
        raise ValueError("FID needs 2 <= N <= 7000")
    metric = build_fid(device)
    classifier, classifier_info = load_model("mlp", cfg, device, data)
    predicted_counts = torch.zeros(10, dtype=torch.long, device=device)
    confidence_sum = 0.0
    indices = np.random.default_rng(cfg["seed"] + 333).permutation(data.indices["test"])[:n]
    output = Path(output_root or cfg["output_dir"]) / "evaluation"
    output.mkdir(parents=True, exist_ok=True)
    index_path = output / f"fid_real_indices_n{n}.npy"
    if index_path.exists() and not np.array_equal(np.load(index_path), indices):
        raise ValueError("Existing FID subset differs")
    np.save(index_path, indices)
    generator = torch.Generator(device=device).manual_seed(cfg["seed"] + 13000)
    batch_size = cfg["fid_batch_size"]
    sync(device)
    start = time.perf_counter()
    for i in range(0, n, batch_size):
        real = data.images[indices[i:i+batch_size]].to(device).float().div(255)
        metric.update(real.repeat(1, 3, 1, 1), real=True)
        noise = make_noise(name, len(real), cfg, device, generator)
        fake = sample(name, model, noise, generator)
        metric.update(fake.repeat(1, 3, 1, 1), real=False)
        probabilities = classifier(fake).softmax(1)
        confidence, prediction = probabilities.max(1)
        predicted_counts += torch.bincount(prediction, minlength=10)
        confidence_sum += confidence.sum().item()
    score = metric.compute().item()
    sync(device)
    return {"FID": score, "n_real": n, "n_synthetic": n, "seconds": time.perf_counter() - start,
            "subset_file": str(index_path), "subset_sha256": hashlib.sha256(index_path.read_bytes()).hexdigest(),
            "feature_dim": 2048, "extractor": "torch-fidelity Inception-v3 via TorchMetrics",
            "preprocessing": "grayscale repeated RGB; float32 [0,1]; normalize=True; extractor handles resize",
            "seed": cfg["seed"] + 13000, "batch_size": batch_size,
            "predicted_label_proxy": {"counts": predicted_counts.cpu().tolist(),
                                      "mean_confidence": confidence_sum / n,
                                      "classifier_sha256": classifier_info["sha256"],
                                      "warning": "Classifier predictions on synthetic/OOD images are not true labels, nor formal precision/recall or proof of mode coverage"},
            "limitation": "ImageNet features on 28x28 grayscale; finite-sample bias, especially N<=2048; not directly comparable to other pipelines"}


In [ ]:
evaluation = {}
if models and (RECOMPUTE_METRICS or RECOMPUTE_FID):
    seed_all(cfg["seed"])
    x, _, _ = to_device(next(iter(data.loader("test"))), device)
    for name in MODEL_NAMES:
        value = {"checkpoint": metadata[name], "environment": environment(device), "split_sha256": data.split_sha256}
        if RECOMPUTE_METRICS:
            if name in RECONSTRUCTORS:
                value["reconstruction"] = reconstruction_metrics(name, models[name], data, device, cfg)
            value["latency"] = latency(name, models[name], cfg, device, x)
        if RECOMPUTE_FID and name in GENERATORS:
            value["fid"] = fid_metrics(name, models[name], data, cfg, device, cfg["fid_samples"], output_root=OUTPUT)
        evaluation[name] = value
        save_json(OUTPUT / "evaluation" / f"{name}.json", value)
    if RECOMPUTE_METRICS:
        if mlp_results is not None:
            mlp_results["vanilla_ae_metrics"] = evaluation["ae"]["reconstruction"]
            mlp_results["frozen_decoder_metrics"] = reconstruction_metrics("decoder", models["decoder"], data, device, cfg, models["mlp"])
            save_json(OUTPUT / "mlp_experiments/results.json", mlp_results)
            display(mlp_results)
        table_cfg = copy.deepcopy(cfg)
        table_cfg["output_dir"] = str(OUTPUT)
        write_table(table_cfg)
    if RECOMPUTE_FID:
        protocols = {(v["fid"]["n_real"], v["fid"]["n_synthetic"], v["fid"]["subset_sha256"])
                     for v in evaluation.values() if "fid" in v}
        assert len(protocols) == 1
    display(evaluation)
elif not models and (RECOMPUTE_METRICS or RECOMPUTE_FID):
    print("Metrik dilewati: perlu checkpoint tervalidasi, tidak menggunakan bobot acak.")

if DISPLAY_SAVED_RESULTS and not (RECOMPUTE_METRICS or RECOMPUTE_FID):
    table = SAVED_RESULTS / "evaluation/comparison.csv"
    if table.exists():
        print("Tabel tersimpan; bukan evaluasi baru:", table)
        with table.open(encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
        columns = ("model", "latent_dim", "seconds_epoch", "ms_batch64", "MSE", "PSNR", "SSIM", "FID", "FID_N")
        header = "| " + " | ".join(columns) + " |"
        divider = "| " + " | ".join("---" for _ in columns) + " |"
        lines = ["| " + " | ".join(str(row[k]) for k in columns) + " |" for row in rows]
        display(Markdown("\n".join([header, divider, *lines])))
    else:
        print("Belum ada tabel metrik tersimpan. Aktifkan evaluasi setelah checkpoint tersedia.")


## 7. Lima visualisasi wajib
Default menampilkan PNG hasil proyek jika tersedia; bukan sampling baru. Untuk menghasilkan ulang,
aktifkan REGENERATE_FIGURES setelah checkpoint lengkap. Fungsi plotting, seed, indeks sampel,
endpoint dan trajectory disalin dari kode proyek. Sumber perhitungan disimpan ke output notebook.


In [ ]:
LABELS = {"ae": "Vanilla AE", "sparse_ae": "Sparse AE", "vae": "VAE", "gan": "Vanilla GAN", "dcgan": "DCGAN", "ddpm": "DDPM"}

def image(ax, tensor, title=None):
    ax.imshow(tensor.detach().cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=11)


def grid(ax, tensor, nrow=4, title=None):
    data = make_grid(tensor.detach().cpu(), nrow=nrow, padding=2, pad_value=1)
    ax.imshow(data.permute(1, 2, 0).numpy(), vmin=0, vmax=1)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=12)


def export(fig, root, name):
    fig.savefig(root / f"{name}.png", dpi=250, bbox_inches="tight")
    fig.savefig(root / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


In [ ]:
VISUALS_READY = bool(models) and REGENERATE_FIGURES and mlp is not None
root, out = Path(cfg["output_dir"]), OUTPUT / "figures"
names = MODEL_NAMES
classifier = models.get("mlp")
if VISUALS_READY:
    seed_all(cfg["seed"])
    out.mkdir(parents=True, exist_ok=True)

def show_figure(filename):
    path = out / f"{filename}.png" if VISUALS_READY else SAVED_RESULTS / "figures" / f"{filename}.png"
    if path.exists() and (VISUALS_READY or DISPLAY_SAVED_RESULTS):
        print("Dihitung ulang:" if VISUALS_READY else "Figur tersimpan, bukan hitung ulang:", path)
        display(DisplayImage(filename=str(path)))
    else:
        print("Figur belum tersedia:", filename, "— lengkapi checkpoint dan aktifkan REGENERATE_FIGURES.")


### 7.1 Rekonstruksi


In [ ]:
if VISUALS_READY:
    with torch.inference_mode():
        selected = data.indices["test"][:8]
        x = data.images[selected].to(device).float().div(255)
        fig, axes = plt.subplots(4, 8, figsize=(10, 5))
        for row, name in enumerate(("original", "ae", "sparse_ae", "vae")):
            values = x if name == "original" else reconstruct(name, models[name], x)
            for col in range(8):
                image(axes[row, col], values[col])
            axes[row, 0].set_ylabel("Original" if row == 0 else LABELS[name], fontsize=11)
        fig.suptitle("Figur 1 — Rekonstruksi citra test yang sama")
        fig.tight_layout()
        export(fig, out, "figure_1_reconstruction")
show_figure("figure_1_reconstruction")


### 7.2 Random sampling


In [ ]:
if VISUALS_READY:
    with torch.inference_mode():
        samples = {}
        fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
        for i, name in enumerate(("vae", "gan", "dcgan", "ddpm")):
            gen = torch.Generator(device=device).manual_seed(cfg["seed"] + 16000)
            noise = make_noise(name, 16, cfg, device, gen)
            samples[name] = sample(name, models[name], noise, gen).cpu()
            grid(axes[i], samples[name], title=LABELS[name])
        fig.suptitle("Figur 2 — Sampling dari prior (16 citra per model)")
        export(fig, out, "figure_2_sampling")
        atomic_torch_save(samples, out / "samples.pt")
show_figure("figure_2_sampling")


### 7.3 Eksperimen MLP


In [ ]:
if VISUALS_READY:
    with torch.inference_mode():
        fig, axes = plt.subplots(2, 2, figsize=(10, 8))
        grid(axes[0, 0], mlp["originals"], 5, "A. Original test (10 citra)")
        grid(axes[0, 1], torch.cat([mlp["class_transpose"], mlp["latent_transpose"]]), 5,
             "B. Transpos kelas (2 baris atas)\nTranspos laten test (2 baris bawah)")
        grid(axes[1, 0], torch.cat([mlp["frozen_reconstruction"], mlp["ae_reconstruction"]]), 5,
             "C. Frozen decoder (2 baris atas)\nVanilla AE laten64 (2 baris bawah)")
        grid(axes[1, 1], mlp["diagonal_gaussian_samples"], 4,
             "D. Gaussian diagonal fitur test\nStatistik test, analisis akhir")
        for ax in axes.flat:
            ax.set_anchor("N")
        fig.suptitle("Figur 3 — Lima tahap eksperimen MLP")
        fig.tight_layout(rect=(0, 0, 1, 0.95))
        export(fig, out, "figure_3_mlp")
show_figure("figure_3_mlp")


### 7.4 Interpolasi


In [ ]:
if VISUALS_READY:
    with torch.inference_mode():
        # Same true test endpoints for AE/VAE; contrasting Fashion-MNIST classes 0 and 9.
        target_classes = (0, 9)
        test_ids = torch.from_numpy(data.indices["test"])
        test_labels = data.labels[test_ids]
        endpoint_ids = [int(test_ids[(test_labels == c).nonzero()[0]]) for c in target_classes]
        endpoints = data.images[endpoint_ids].to(device).float().div(255)
        alpha = torch.linspace(0, 1, 6, device=device)[:, None]
        interpolation_meta = {"classes": target_classes, "test_endpoint_indices": endpoint_ids,
                              "alphas": alpha.flatten().tolist(), "gan_selection_seed": cfg["seed"] + 17000,
                              "gan_selection_method": "2048 fixed prior candidates, select maximum MLP confidence for predicted classes 0 and 9; requires visual verification, classifier is only proxy"}
        fig, axes = plt.subplots(5, 6, figsize=(9, 7))
        interpolation_tensors = {}
        for row, name in enumerate(("ae", "sparse_ae", "vae", "gan", "dcgan")):
            model = models[name]
            if name in ("ae", "sparse_ae"):
                z = model.encoder(endpoints)
            elif name == "vae":
                z = model.encode(endpoints)[0]
            else:
                gen = torch.Generator(device=device).manual_seed(cfg["seed"] + 17000)
                candidate_z = make_noise(name, 2048, cfg, device, gen)
                probabilities = []
                for batch in candidate_z.split(128):
                    probabilities.append(classifier(sample(name, model, batch)).softmax(1))
                probs = torch.cat(probabilities)
                predicted = probs.argmax(1)
                selected_candidates, confidence = [], []
                for c in target_classes:
                    valid = (predicted == c).nonzero().flatten()
                    if len(valid) == 0:
                        raise RuntimeError(f"{name} has no candidate predicted class {c}; inspect coverage, do not fabricate endpoints")
                    best = valid[probs[valid, c].argmax()]
                    selected_candidates.append(int(best))
                    confidence.append(float(probs[best, c]))
                z = candidate_z[selected_candidates]
                interpolation_meta[name] = {"candidate_indices": selected_candidates, "mlp_confidence": confidence,
                                             "visual_verification": "pending human inspection"}
            intermediate = (1 - alpha) * z[:1] + alpha * z[1:]
            values = model.decoder(intermediate) if name in ("ae", "sparse_ae", "vae") else sample(name, model, intermediate)
            interpolation_tensors[name] = values.cpu()
            for col in range(6):
                image(axes[row, col], values[col], f"α={col/5:.1f}" if row == 0 else None)
            axes[row, 0].set_ylabel(LABELS[name], fontsize=11)
        fig.suptitle("Figur 4 — Interpolasi linear; endpoint kelas 0 vs 9")
        fig.tight_layout()
        export(fig, out, "figure_4_interpolation")
        save_json(out / "interpolation.json", interpolation_meta)
        atomic_torch_save(interpolation_tensors, out / "interpolation.pt")
show_figure("figure_4_interpolation")


### 7.5 Kurva dan reverse trajectory


In [ ]:
if VISUALS_READY:
    with torch.inference_mode():
        fig = plt.figure(figsize=(12, 10))
        layout = fig.add_gridspec(4, 6)
        for i, name in enumerate(names):
            ax = fig.add_subplot(layout[i // 2, (i % 2)*3:(i % 2)*3+3])
            history = json.loads((root / name / "history.json").read_text(encoding="utf-8"))
            epoch = [h["epoch"] for h in history]
            if name in ("gan", "dcgan"):
                for key, label in (("train_g_loss", "G"), ("train_d_loss", "D")):
                    ax.plot(epoch, [h[key] for h in history], label=label)
            else:
                ax.plot(epoch, [h["train_loss"] for h in history], label="train objective")
                ax.plot(epoch, [h["val_loss"] for h in history], label="validation objective")
                if name == "vae":
                    ax.plot(epoch, [h["train_recon_bce_sum"] for h in history], label="BCE")
                    ax.plot(epoch, [h["train_kl_sum"] for h in history], label="KL")
            ax.set_title(LABELS[name], fontsize=14)
            ax.set_xlabel("epoch", fontsize=12)
            ax.set_ylabel("loss (model-specific)", fontsize=11)
            ax.tick_params(labelsize=11)
            ax.legend(fontsize=11)
            ax.grid(alpha=0.2)
        gen = torch.Generator(device=device).manual_seed(cfg["seed"] + 18000)
        noise = make_noise("ddpm", 1, cfg, device, gen)
        _, chain = models["ddpm"].sample(noise, gen, snapshots=True)
        for col, (t, state) in enumerate(chain.items()):
            ax = fig.add_subplot(layout[3, col])
            image(ax, state.add(1).div(2).clamp(0, 1), f"x_{t}")
        fig.suptitle("Figur 5 — Kurva loss dan satu reverse chain DDPM")
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        export(fig, out, "figure_5_curves_trajectory")
        atomic_torch_save(chain, out / "ddpm_trajectory.pt")
        save_json(out / "provenance.json", {"checkpoints": metadata, "split_sha256": data.split_sha256,
                                            "figure_1_test_indices": selected.tolist(),
                                            "sampling_seed": cfg["seed"] + 16000, "sampling_indices": list(range(16)),
                                            "trajectory_seed": cfg["seed"] + 18000})
show_figure("figure_5_curves_trajectory")


## Catatan akhir
- Checkpoint/split/seed sama tidak menjamin bitwise lintas perangkat atau versi pustaka.
- Jangan menyebut smoke sebagai training penuh atau angka tersimpan sebagai penghitungan baru.
- Statistik laten test hanya untuk tahap5; bukan tuning, pemilihan checkpoint, atau FID generator.
- FID ImageNet pada grayscale28×28, satu seed, dan budget training berbeda membatasi generalisasi.
- Catatan bantuan AI dan sumber modular tetap dipertahankan; notebook merupakan penyusunan ulang,
  bukan klaim bahwa implementasi sebelumnya tanpa bantuan. Kelas LA01; tinjauan mahasiswa masih menunggu.
- Notebook tidak mencakup pembangun PDF/publikasi GitHub: tujuh bagian eksperimen di atas berdiri sendiri.
